In [ ]:
"""
Excluding Experimental+AlSi10Mg literature
Random Forest Training + Prediction Script
Python 3.10
Dependencies: pandas, numpy, scikit-learn, joblib
"""

import pandas as pd
import numpy as np
import joblib
import warnings
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

# -------------------------------
# 1. Load dataset
# -------------------------------
dataset_path = r"C:\Users\gurra\Downloads\Excluding_30_AlSi10Mg.csv"
dataset = pd.read_csv(dataset_path)

# -------------------------------
# 2. Features & Target
# -------------------------------
features = [
    'power','speed','thickness','dia','Solidt','Sdensity','Sspheat','Sthercondu',
    'Liquidt','Ldensity','LSpheat','Lthercondu','Lsurfacet','Lviscosity',
    'Lfusion','dsigma','Absorptivity','Lvapor'
]
X = dataset[features]
y = dataset['defect']

# -------------------------------
# 3. Scale features
# -------------------------------
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.23, random_state=42)

# -------------------------------
# 4. Random Forest + GridSearchCV
# -------------------------------
param_grid = {
    "n_estimators": [50, 75, 100, 150, 200],
    "max_depth": [4, 5, 6, 7, 8, None]
}

rf_model = RandomForestClassifier(random_state=42)

grid = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print("\nBest Parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

# -------------------------------
# 5. Evaluate model
# -------------------------------
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

print("\nTrain Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nConfusion Matrix (Test Set):\n", confusion_matrix(y_test, y_test_pred))
print("\nClassification Report (Test Set):\n", classification_report(y_test, y_test_pred, labels=[0,1,2,3]))

# -------------------------------
# 6. Save model & scaler
# -------------------------------
joblib.dump(best_model, "RF_model.pkl")
joblib.dump(scaler, "RF_scaler.pkl")

# -------------------------------
# 7. Predict new alloy dataset
# -------------------------------
new_data_path = r"C:\Users\gurra\Downloads\final 5-800w power speed Al alloy predictions.csv"
new_data = pd.read_csv(new_data_path)

# Scale features
X_new_scaled = scaler.transform(new_data[features])

# Predictions & probabilities
y_new_pred = best_model.predict(X_new_scaled)
y_new_proba = best_model.predict_proba(X_new_scaled)

# Map numeric predictions to labels
class_map = {0: 'Good', 1: 'Balling', 2: 'Lack of fusion', 3: 'Keyholing'}
new_data['Predicted_class'] = y_new_pred
new_data['Predicted_Label'] = new_data['Predicted_class'].map(class_map)

# Add probability columns
proba_df = pd.DataFrame(
    y_new_proba,
    columns=[f"Prob_{class_map[c]}" for c in sorted(class_map.keys())]
)
new_data = pd.concat([new_data, proba_df], axis=1)

# Save predictions
output_file = r"C:\Users\gurra\Downloads\EE_Al_RF_predictions.csv"
new_data.to_csv(output_file, index=False)
print(f"\nPredictions saved to: {output_file}")

# Save model & scaler
joblib.dump(best_model, r"C:\Users\gurra\Downloads\RF_model.pkl")
joblib.dump(scaler, r"C:\Users\gurra\Downloads\RF_scaler.pkl")


Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best Parameters: {'max_depth': 7, 'n_estimators': 75}
Best CV Accuracy: 0.7834586466165414

Train Accuracy: 0.9609929078014184
Test Accuracy: 0.788235294117647

Confusion Matrix (Test Set):
 [[16  1  4  3]
 [ 0 13  1  1]
 [ 0  1 20  1]
 [ 2  3  1 18]]

Classification Report (Test Set):
               precision    recall  f1-score   support

           0       0.89      0.67      0.76        24
           1       0.72      0.87      0.79        15
           2       0.77      0.91      0.83        22
           3       0.78      0.75      0.77        24

    accuracy                           0.79        85
   macro avg       0.79      0.80      0.79        85
weighted avg       0.80      0.79      0.79        85


Predictions saved to: C:\Users\gurra\Downloads\EE_718_RF_predictions.csv


['C:\\Users\\gurra\\Downloads\\RF_scaler.pkl']